In [ ]:
# ── CELL 1: Mount Drive and clone repo ────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository
# !git clone https://github.com/Anush-K/Retinal_SSL.git
%cd Retinal_SSL

Cloning into 'Retinal_SSL'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23): 100% (23/23), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 23 (delta 1), reused 23 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 31.20 KiB | 2.84 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/Users/anushk/Desktop/Retinal_SSL/Retinal_SSL


In [ ]:
# ── CELL 2: Install dependencies ──────────────────────────────
!pip install -r requirements.txt -q

In [ ]:
# ── CELL 3: Prepare dataset (run ONCE only) ───────────────────
# After first run, comment this cell out
!python prepare_dataset.py

In [ ]:
# ── CELL 4: Verify dataset preparation ────────────────────────
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/APTOS_metadata.csv")
print(f"Total samples: {len(df)}")
for split in ["train", "val", "test"]:
    sub = df[df["image_path"].str.contains(f"/{split}/")]
    print(f"  {split}: {(sub.label==0).sum()} NORMAL, "
          f"{(sub.label==1).sum()} ABNORMAL")

In [ ]:
# ── CELL 5: SSL Pretraining ────────────────────────────────────
!python train/train_ssl.py

In [ ]:
# ── CELL 6: t-SNE Visualization ───────────────────────────────
!python train/run_tsne.py

In [ ]:
# ── CELL 7: Fine-tune WITH SSL weights (DBFC — main result) ───
!python train/train_finetune.py \
    --run_label "DBFC_SSL" \
    --results_dir "/content/drive/MyDrive/results_ssl_dbfc" \
    --ssl_checkpoint "/content/drive/MyDrive/checkpoints/ssl_final.pth"


In [ ]:
# ── CELL 8: Fine-tune WITHOUT SSL (ImageNet baseline) ─────────
!python train/train_finetune_no_ssl.py \
    --run_label "No_SSL" \
    --results_dir "/content/drive/MyDrive/results_nossl"

In [ ]:
!python train/print_ablation.py

In [ ]:
# ── CELL 10: Display all result images ────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

result_folders = [
    "/content/drive/MyDrive/results_ssl_dbfc",
    "/content/drive/MyDrive/results_nossl",
]

for folder in result_folders:
    pngs = sorted(glob.glob(f"{folder}/*.png"))
    for p in pngs:
        img = mpimg.imread(p)
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.title(p.split("/")[-1])
        plt.axis("off")
        plt.tight_layout()
        plt.show()